# nexus-lite × Colab A100 (Pro): heterogeneous local-LLM swarm

Runtime → **A100 (Colab Pro)**. 40GB VRAM → Qwen 3.8 27B (≈18GB) + Gemma 4 12B (≈8GB) loaded together.
Runs Ollama on the Colab GPU and drives a parallel swarm + sequential pipeline of *local* model agents via `nexus_lite.run_swarm` / `run_pipeline`. No API keys.

## 1. GPU check + install

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!apt-get update -qq && apt-get install -y -qq dtach zstd
!pip install -q git+https://github.com/choilab-jefferson/nexus.lite.git
!curl -fsSL https://ollama.com/install.sh | sh

## 2. Start Ollama (GPU) and pull model(s)

In [ ]:
import subprocess, time, urllib.request, os
MODELS = ['qwen3.8:27b', 'gemma4:12b']
MODEL = MODELS[0]
os.environ.update({'OLLAMA_NUM_PARALLEL': '3', 'OLLAMA_MAX_LOADED_MODELS': '2'})
subprocess.Popen(["ollama", "serve"], stdout=open("/tmp/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try: urllib.request.urlopen('http://127.0.0.1:11434'); break
    except Exception: time.sleep(1)
for m in MODELS:
    subprocess.run(['ollama', 'pull', m], check=True)
!ollama list

## 3. Agent wrapper
`run_swarm` feeds each spec's `prompt` to its `cmd` on **stdin**; this helper (also `examples/ollama_agent.py`) forwards it to the local model.

In [ ]:
import pathlib
pathlib.Path('/content/nexus-agents').mkdir(exist_ok=True)
pathlib.Path('/content/nexus-agents/ollama_agent.py').write_text('#!/usr/bin/env python3\n"""stdin -> local Ollama (GPU) -> stdout. Usable as a nexus-lite `cmd` agent.\n\nUsage: echo "prompt" | python3 ollama_agent.py [--model qwen2.5:3b] [--system TEXT]\n"""\nimport argparse, json, re, sys, urllib.request\n\nap = argparse.ArgumentParser()\nap.add_argument("--model", default="gemma4:12b")\nap.add_argument("--system", default="")\nap.add_argument("--host", default="http://127.0.0.1:11434")\nap.add_argument("--max-tokens", type=int, default=600)\na = ap.parse_args()\n\nbody = {"model": a.model, "prompt": sys.stdin.read(), "system": a.system,\n        "stream": False, "options": {"num_predict": a.max_tokens, "temperature": 0.3}}\nreq = urllib.request.Request(a.host + "/api/generate", json.dumps(body).encode(),\n                             {"Content-Type": "application/json"})\ntry:\n    out = json.load(urllib.request.urlopen(req, timeout=600))["response"]\n    print(re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip())\nexcept Exception as e:\n    print(f"ollama_agent error: {e}", file=sys.stderr)\n    sys.exit(1)\n')
def agent(system, model=None):
    return f'python3 /content/nexus-agents/ollama_agent.py --model {model or MODEL} --system "{system}"'
print(subprocess.run(agent('Answer in one sentence.'), shell=True, input='What is a GPU?', text=True, capture_output=True).stdout)

## 4. Parallel swarm: 3 local agents on one GPU
Heterogeneous: the security reviewer runs on `gemma4:12b`, the others on `qwen3.8:27b` (both stay resident in VRAM).

In [ ]:
import nexus_lite as nl
nl.ensure_colab()
topic = 'a REST API for a todo-list service'
specs = [
  {'task_id': 'arch',  'cmd': agent('You are a software architect. Be concise.', None), 'prompt': f'List 4 key design decisions for {topic}.'},
  {'task_id': 'sec',   'cmd': agent('You are a security reviewer. Be concise.', 'gemma4:12b'),  'prompt': f'List 4 security risks for {topic}.'},
  {'task_id': 'tests', 'cmd': agent('You are a QA engineer. Be concise.', None),        'prompt': f'List 4 test cases for {topic}.'},
]
t0 = time.time()
results = nl.run_swarm(specs, caller='orchestrator', timeout=600.0)
print(f'swarm finished in {time.time()-t0:.1f}s')
for r in results: print(f"\n[{r['status'].upper()}] {r['task_id']}\n{r['output'].strip()}")

## 5. Sequential pipeline: driver → verifier → documenter

In [ ]:
stages = [
  {'name': 'driver',     'cmd': agent('You write short Python code.'),          'prompt': 'Write a Python function slugify(text). Output only code.'},
  {'name': 'verifier',   'cmd': agent('You review code for bugs. Be concise.', 'gemma4:12b'), 'prompt': 'Review this code and list issues:\n{prev_output}'},
  {'name': 'documenter', 'cmd': agent('You write concise docs.'),               'prompt': 'Write a 3-line docstring based on this review:\n{prev_output}'},
]
pipe = nl.run_pipeline(stages, initial_prompt='slugify', caller='orchestrator', timeout=600.0)
for s in pipe['stages']: print(f"--- {s['stage']} (rc={s.get('rc')}) ---\n{s.get('output','').strip()}\n")

## 6. Inspect: GPU utilisation, loaded models, dashboard

In [ ]:
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv
!ollama ps
nl.dashboard()